<a href="https://colab.research.google.com/github/priyu9-star/BudgetWise-AI-based-Expense-Forecasting-Tool-Batch-6-Team-C-/blob/main/login_registrationPage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import os
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)


In [13]:
%%writefile templates/login.html
<!DOCTYPE html>
<html>
<head>
    <title>Login - BudgetWise</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
    <h2>Login</h2>
    <form method="POST">
        <input name="email" type="email" placeholder="Email" required>
        <input name="password" type="password" placeholder="Password" required>
        <button type="submit">Login</button>
    </form>

    {% if error %}
    <p class="error">{{ error }}</p>
    {% endif %}

    <p>Don’t have an account? <a href="/register">Register</a></p>
</div>
</body>
</html>


Overwriting templates/login.html


In [14]:
%%writefile templates/register.html
<!DOCTYPE html>
<html>
<head>
    <title>Register - BudgetWise</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="container">
    <h2>Create Account</h2>
    <form method="POST">
        <input name="username" placeholder="Username" required>
        <input name="email" type="email" placeholder="Email" required>
        <input name="password" type="password" placeholder="Password" required>
        <button type="submit">Register</button>
    </form>

    <p>Already have an account? <a href="/login">Login</a></p>
</div>
</body>
</html>


Overwriting templates/register.html


In [15]:
%%writefile static/style.css
body{
    margin:0;
    font-family:'Poppins',sans-serif;
    height:100vh;
    background-color:#B37075;
    display:flex;
    justify-content:center;
    align-items:center;
}
.container{
    width:350px;
    padding:30px;
    background:rgba(255,255,255,0.15);
    border-radius:16px;
    backdrop-filter:blur(8px);
    text-align:center;
    color:white;
}
input,button{
    width:100%;
    padding:14px;
    margin:10px 0;
    border-radius:10px;
    border:none;
}
button{
    background:white;
    color:black;
    font-weight:bold;
}
.error{color:yellow;font-weight:bold;}

Overwriting static/style.css


In [16]:
import os
!pip install flask-sqlalchemy flask-bcrypt
from flask import Flask, render_template, request, redirect, session, jsonify, render_template_string
from flask_sqlalchemy import SQLAlchemy
from flask_bcrypt import Bcrypt

app = Flask(__name__)
app.secret_key = "secret123"

app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///users.db"
db = SQLAlchemy(app)
bcrypt = Bcrypt(app)

# ---------------- USER MODEL ----------------
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80))
    email = db.Column(db.String(120), unique=True)
    password = db.Column(db.String(200))

    def check_pass(self, pwd):
        return bcrypt.check_password_hash(self.password, pwd)

with app.app_context():
    db.create_all()


# ---------------- ROUTES ----------------

# ROOT → LOGIN PAGE
@app.route("/")
def root():
    return redirect("/login")


@app.route("/login", methods=["GET","POST"])
def login():
    if request.method == "POST":
        email = request.form["email"]
        pwd = request.form["password"]

        user = User.query.filter_by(email=email).first()
        if user and user.check_pass(pwd):
            session["user_id"] = user.id
            return redirect("/dashboard")   # ❤️ DIRECTLY GO TO YOUR DASHBOARD
        return render_template("login.html", error="Invalid credentials")

    return render_template("login.html")


@app.route("/register", methods=["GET","POST"])
def register():
    if request.method == "POST":
        username = request.form["username"]
        email = request.form["email"]
        password = bcrypt.generate_password_hash(request.form["password"]).decode("utf-8")

        u = User(username=username, email=email, password=password)
        db.session.add(u)
        db.session.commit()

        return redirect("/login")

    return render_template("register.html")


@app.route("/logout")
def logout():
    session.clear()
    return redirect("/login")


# -------------- PROTECTION FOR DASHBOARD ---------------
@app.before_request
def protect_dashboard():
    if request.path.startswith("/dashboard"):
        if "user_id" not in session:
            return redirect("/login")

In [17]:
@app.route('/dashboard')
def dashboard():
    return render_template_string(dashboard_html)


In [18]:
!pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("35OHwrUOnBO4DwOnuX7Yp4RFJBH_62Nddc9w2BgJ3kyr3acXy") # Replace YOUR_AUTH_TOKEN_HERE with your actual ngrok authtoken

public_url = ngrok.connect(5000)
print("Public URL:", public_url)

app.run(port=5000)

Public URL: NgrokTunnel: "https://unpunctual-turbanlike-alicia.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:25] "GET / HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:26] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:26] "GET /static/style.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:44] "GET /register HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:45] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:49] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:50] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:53] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 13:37:54] "GET /static/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [19/Nov/2025 1